# Script process embeddings

In [1]:
import pandas as pd 
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os

# Get absolute path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

from utils.f_utils import load_config, load_models, preprocess_caption, load_dataset
from prompts.prompt_builder import PromptBuilderFactory, select_few_shot_examples
from data.ImageClef_train_embeddings import ImageClefDataset, generate_image_embedding

config = load_config("../configs/few_shot_prompt.yaml")
if config.get("medsiglip"):
    processor, model, device = load_models(config, device='cuda')

In [3]:
# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it", token = os.environ["HUGGINGFACE_HUB_TOKEN_WRITE"])
model = AutoModelForImageTextToText.from_pretrained("google/medgemma-1.5-4b-it", token = os.environ["HUGGINGFACE_HUB_TOKEN_WRITE"])
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

config.json: 0.00B [00:00, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


The animal on the candy is a **bird**.
<end_of_turn>


In [12]:
import json

n_few_shot = 3
train_dataset = None
if n_few_shot > 0 or prompt_strategy == "few_shot":
    with open("/home/ia368/projetos/imageclef2026-rag/artifacts/datasets/imageclef2026_train_dataset.json", 'r') as file:
    # Use json.load() to parse the file content into a Python object
        train_dataset = ImageClefDataset(json.load(file))

In [13]:
select_few_shot_examples(train_dataset, n_few_shot)

[{'image': <PIL.Image.Image image mode=RGB size=750x552>,
  'caption': 'RA diastolic collapse prior to pericardiocentesis.',
  'id': 'ImageCLEFmedical_Caption_2026_train_5342',
  'concepts': None},
 {'image': <PIL.Image.Image image mode=RGB size=752x752>,
  'caption': 'Pretreatment magnetic resonance imaging of our patient with endometriosis. Axial T1-weighted imaging at the first consultation. Two ovarian tumors (tumor size: right < left) show high-intensity signals (arrows), and the internal structure shows a blood-resistant component.',
  'id': 'ImageCLEFmedical_Caption_2026_train_31894',
  'concepts': None},
 {'image': <PIL.Image.Image image mode=RGB size=732x714>,
  'caption': 'Angiography approached from the proximal portion of the right hepatic artery indicated a shunt in the pulmonary circulation system. A catheter for chemoembolization was placed distally from the shunt for doxorubicin infusion.',
  'id': 'ImageCLEFmedical_Caption_2026_train_4071',
  'concepts': None}]

In [2]:
ds = load_dataset(config)

In [3]:

class ImageCLEF24_Dataset(Dataset):
    def __init__(self, split, processor, max_length=64):
        """
        split: split do dataset (ex: ds["train"])
        processor: processor do MedSigLip
        max_length: tamanho máximo do texto
        """
        self.dataset = split
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        # Carrega a imagem apenas agora
        image = Image.open(sample["image_path"]).convert('RGB')
        text = sample["caption"]

        # Usa o processor do modelo
        # texto (caption) + imagem -> embedding multimodal do MedSigLip ?
        encoding = self.processor(
            text=text,
            images=image,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Remove batch dimension criada pelo return_tensors="pt"
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        return encoding


In [4]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import argparse
import sys
# Get absolute path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)
from utils.f_utils import load_config, load_models, load_dataset
from PIL import Image

dataset = ImageCLEF24_Dataset(
        split=ds[0:10],
        processor=processor,
        max_length=64
    )

dataloader = DataLoader(
    dataset,
    batch_size=config['parameters']['batch_size'],
    shuffle=False,
    pin_memory=True
)

print(f"Using device: {device}")


def generate_embeddings(model, dataloader, device):
    model.eval()
    model.to(device)

    all_image_embeds = []
    all_text_embeds = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Generating embeddings"):
            batch = {k: v.to(device) for k, v in batch.items()}

            image_embeds = model.get_image_features(
                pixel_values=batch["pixel_values"]
            )

            text_embeds = model.get_text_features(
                input_ids=batch["input_ids"]
            )

            # normalização (cosine similarity)
            image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
            text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

            all_image_embeds.append(image_embeds.cpu())
            all_text_embeds.append(text_embeds.cpu())

    all_image_embeds = torch.cat(all_image_embeds, dim=0)
    all_text_embeds = torch.cat(all_text_embeds, dim=0)

    return all_image_embeds, all_text_embeds


image_embeds, text_embeds = generate_embeddings(
    model,
    dataloader,
    device
)

print("Saving embeddings...")

OUTPUT_DIR = config['output_dir']
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.save(image_embeds, os.path.join(OUTPUT_DIR, f"{config['output_name']}_image_embeddings.pt"))
torch.save(text_embeds, os.path.join(OUTPUT_DIR, f"{config['output_name']}_text_embeddings.pt"))

# também salva captions alinhadas
captions = [sample["caption"] for sample in ds]
torch.save(captions, os.path.join(OUTPUT_DIR, f"{config['output_name']}_captions.pt"))

print("Done!")
print(f"Saved to {OUTPUT_DIR}")

Using device: cuda


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]

Saving embeddings...
Done!
Saved to /home/ia368/projetos/imageclef2026-rag/artifacts//embeddings


# Load embeddings

In [5]:
config_vs = load_config("../configs/IC2024_vector_store.yaml")
config_vs

{'folders': {'embeddings_dir': '/home/ia368/projetos/imageclef2026-rag/artifacts/embeddings',
  'vector_store_dir': '/home/ia368/projetos/imageclef2026-rag/artifacts/vector_store'},
 'files': {'captions': 'IC2024_medsiglip_train_captions.pt',
  'image': 'IC2024_medsiglip_train_image_embeddings.pt',
  'text': 'IC2024_medsiglip_train_text_embeddings.pt'}}

In [6]:
def load_embeddings(config, type_='image'):
    """
    Carrega arquivo .pt
    """
    
    emb_file = os.path.join(config['folders']['embeddings_dir'], config['files'][type_])
    
    if not os.path.exists(emb_file):
        print(f"❌ Arquivo não encontrado: {emb_file}")
        return None
    
    try:
        # Carregar o arquivo torch
        embeddings = torch.load(emb_file)
        
        print(f"✅ Embeddings carregados com sucesso!")
        print(f"📊 Formato dos dados: {type(embeddings)}")
        print(f"📊 Shape: {embeddings.shape if hasattr(embeddings, 'shape') else 'N/A'}")
        
        return embeddings
        
    except Exception as e:
        print(f"❌ Erro ao carregar embeddings: {e}")
        return None

In [7]:
embs = load_embeddings(config_vs)

embs.shape[0]

✅ Embeddings carregados com sucesso!
📊 Formato dos dados: <class 'torch.Tensor'>
📊 Shape: torch.Size([70108, 1152])


70108

In [2]:
import torch
torch.cuda.is_available()

True

# Tenta carregar o MedGemma

In [1]:
"""
MedGemma model loading and configuration.
"""
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from typing import Tuple, Dict, Any, Optional


def check_gpu_capability() -> bool:
    """
    Check if GPU supports bfloat16.
    
    Returns:
        True if GPU supports bfloat16, False otherwise
        
    Raises:
        ValueError: If GPU doesn't support bfloat16
    """
    if not torch.cuda.is_available():
        raise ValueError("CUDA is not available. This model requires a GPU.")
    
    if torch.cuda.get_device_capability()[0] < 8:
        raise ValueError("GPU does not support bfloat16, please use a GPU that supports bfloat16.")
    
    return True


def load_medgemma_model(
    model_id: str = "google/medgemma-4b-it",
    use_quantization: bool = True,
    attn_implementation: str = "eager",
) -> Tuple[Any, Any]:
    """
    Load MedGemma model and processor.
    
    Args:
        model_id: Hugging Face model identifier
        use_quantization: Whether to use 4-bit quantization
        attn_implementation: Attention implementation ("eager" or "flash_attention_2")
        
    Returns:
        Tuple of (model, processor)
    """
    check_gpu_capability()
    
    dtype = torch.bfloat16
    model_kwargs: Dict[str, Any] = dict(
        attn_implementation=attn_implementation,
        dtype=dtype,  # Use dtype instead of deprecated torch_dtype
        device_map="auto",
    )
    
    if use_quantization:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_quant_storage=dtype,
        )
    
    model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
    processor = AutoProcessor.from_pretrained(model_id, use_fast=False)  # Keep slow processor for original behavior
    
    # Use right padding to avoid issues during training
    processor.tokenizer.padding_side = "right"
    
    return model, processor

In [3]:
model, processor = load_medgemma_model(
    model_id="google/medgemma-4b-it",
    use_quantization=True,
)
print("   ✓ Model and processor loaded successfully")
print(f"   Model device: {next(model.parameters()).device}")
print(f"   Model dtype: {next(model.parameters()).dtype}")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

   ✓ Model and processor loaded successfully
   Model device: cuda:0
   Model dtype: torch.bfloat16


In [4]:
# pip install accelerate
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import requests
import torch


# Image attribution: Stillwaterising, CC0, via Wikimedia Commons
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"
image = Image.open(requests.get(image_url, headers={"User-Agent": "example"}, stream=True).raw)

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are an expert radiologist."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this X-ray"},
            {"type": "image", "image": image}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)


Okay, based on the image provided, here's a description of the X-ray:

**Overall Impression:**

The X-ray shows a standard chest view, demonstrating the lungs, heart, major vessels, and bony structures of the chest wall. The image is well-exposed and shows good visualization of the mediastinum and lung fields.

**Specific Findings:**

*   **Lungs:** The lungs appear clear, with no obvious consolidation, nodules, or masses. There is no evidence of significant pulmonary edema or pleural effusion.
*   **Heart:** The heart size appears within normal limits. The cardiomediastinal silhouette is unremarkable.
*   **Mediastinum:** The mediastinum is unremarkable, with no obvious masses or enlarged lymph nodes.
*   **Bones:** The ribs and clavicles appear intact. There is no evidence of fractures or significant degenerative changes.
*   **Soft Tissues:** The soft tissues of the chest wall are unremarkable.
